# 配套实践 08-01：循环记忆怎样形成

本练习先构造两段当前观测相同、历史过程不同的机器人轨迹，再用 NumPy 手工展开一个最小 RNN。我们将观察隐藏状态怎样保存历史、顺序改变为什么会改变结果，以及 mask 怎样阻止补齐位置继续更新记忆。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/08-time-and-memory/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 处理轨迹、矩阵运算和循环隐藏状态
import matplotlib.pyplot as plt  # 绘制输入历史、隐藏状态和最终记忆
np.random.seed(81)  # 固定人工轨迹中的随机扰动以便重复观察
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像的显示清晰度
plt.rcParams["axes.unicode_minus"] = False  # 避免负号在部分字体中显示异常

## 1. 当前相同，不代表过程相同

每个时间步包含两个量：末端相对杯子的高度，以及触觉接触强度。第一段轨迹逐步接近杯子但尚未接触；第二段轨迹先出现接触，随后释放并离开。我们让两段轨迹在最后一步拥有相同数值，以突出历史信息的作用。

In [ ]:
time_steps = np.arange(12)  # 建立十二个离散时间步
approach_height = np.linspace(0.30, 0.08, len(time_steps))  # 构造逐步下降的接近轨迹
release_height = np.concatenate([np.linspace(0.08, 0.03, 5), np.linspace(0.03, 0.08, 7)])  # 构造先接触后离开的轨迹
approach_touch = np.zeros_like(approach_height)  # 接近过程还没有出现接触
release_touch = 0.85 * np.exp(-0.5 * ((time_steps - 4.0) / 1.2) ** 2)  # 在释放前构造一次短暂接触峰值
approach_height[-1] = 0.08  # 令第一段轨迹的最终高度等于共同当前观测
release_height[-1] = 0.08  # 令第二段轨迹的最终高度等于共同当前观测
release_touch[-1] = 0.0  # 令第二段轨迹的最终触觉也回到零
approach_sequence = np.stack([approach_height, approach_touch], axis=-1)  # 合并第一段轨迹的高度和触觉
release_sequence = np.stack([release_height, release_touch], axis=-1)  # 合并第二段轨迹的高度和触觉
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True)  # 创建高度和触觉两个并排坐标轴
axes[0].plot(time_steps, approach_height, "o-", label="Approach")  # 绘制逐步接近时的末端高度
axes[0].plot(time_steps, release_height, "s-", label="Contact then release")  # 绘制接触后离开时的末端高度
axes[0].set(title="End-effector height", xlabel="Time step", ylabel="Height / m")  # 使用通用英文字体标注高度曲线的物理含义
axes[0].legend()  # 显示两段历史的图例
axes[1].plot(time_steps, approach_touch, "o-", label="Approach")  # 绘制尚未接触时的触觉信号
axes[1].plot(time_steps, release_touch, "s-", label="Contact then release")  # 绘制短暂出现后消失的接触峰值
axes[1].set(title="Tactile contact", xlabel="Time step", ylabel="Normalized intensity")  # 使用通用英文字体标注触觉曲线的含义
axes[1].legend()  # 显示两段触觉历史的图例
fig.suptitle("Same current observation, different histories")  # 使用通用英文字体强调本节需要观察的现象
fig.tight_layout()  # 自动调整子图间距避免文字重叠
plt.show()  # 显示两段历史的对比图

**怎样理解结果：** 两条高度曲线最终都到达 0.08 m，最后一步触觉也都为零。如果模型只读取最后一步，它无法区分“仍在接近”和“已经接触后离开”。第二条轨迹中间的触觉峰值以及高度变化方向，才说明此前发生过什么。

## 2. 手工展开一个最小 RNN

下面固定一组容易观察的权重，不进行训练。每一步都把当前高度、触觉信号和上一时刻隐藏状态放进同一个更新式。隐藏状态有两个维度，它们没有预先规定的物理名称，只是用于观察不同历史怎样留下不同数值痕迹。

In [ ]:
input_weights = np.array([[2.8, 1.4], [-1.6, 2.2]])  # 定义当前输入到两个隐藏维度的权重
hidden_weights = np.array([[0.72, 0.10], [0.05, 0.80]])  # 定义旧隐藏状态继续影响新状态的权重
hidden_bias = np.array([-0.28, 0.02])  # 设置隐藏状态更新时使用的偏置
def run_manual_rnn(inputs, valid_mask=None):  # 定义沿时间逐步更新隐藏状态的函数
    if valid_mask is None:  # 判断调用者是否提供了有效时间步掩码
        valid_mask = np.ones(len(inputs), dtype=bool)  # 未提供时默认所有时间步都是真实观测
    hidden = np.zeros(2, dtype=np.float64)  # 使用全零向量初始化循环记忆
    hidden_history = []  # 创建列表保存每一步更新后的隐藏状态
    for index, current_input in enumerate(inputs):  # 按原始时间顺序读取每一个输入
        candidate = np.tanh(input_weights @ current_input + hidden_weights @ hidden + hidden_bias)  # 根据当前输入和旧记忆计算候选新状态
        hidden = candidate if valid_mask[index] else hidden  # 真实时间步更新记忆而补齐位置保持旧状态
        hidden_history.append(hidden.copy())  # 保存副本避免后续更新覆盖此前结果
    return np.stack(hidden_history, axis=0)  # 把隐藏状态列表整理成时间乘隐藏维度的数组
approach_hidden = run_manual_rnn(approach_sequence)  # 计算接近历史对应的隐藏状态序列
release_hidden = run_manual_rnn(release_sequence)  # 计算接触后离开历史对应的隐藏状态序列

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)  # 创建两个坐标轴分别显示两段历史的内部记忆
for hidden_index in range(2):  # 依次绘制两个隐藏维度
    axes[0].plot(time_steps, approach_hidden[:, hidden_index], marker="o", label=f"Hidden {hidden_index + 1}")  # 绘制接近过程的隐藏状态变化
    axes[1].plot(time_steps, release_hidden[:, hidden_index], marker="s", label=f"Hidden {hidden_index + 1}")  # 绘制接触后离开过程的隐藏状态变化
axes[0].set(title="Memory after approach", xlabel="Time step", ylabel="Hidden-state value")  # 使用通用英文字体标注第一幅隐藏状态图
axes[1].set(title="Memory after contact and release", xlabel="Time step")  # 使用通用英文字体标注第二幅隐藏状态图
axes[0].legend()  # 显示第一幅图的隐藏维度图例
axes[1].legend()  # 显示第二幅图的隐藏维度图例
fig.suptitle("The same current input can produce different memories")  # 使用通用英文字体强调循环记忆保留了历史差异
fig.tight_layout()  # 调整图像布局避免标题相互遮挡
plt.show()  # 显示隐藏状态随时间变化的结果

**怎样理解结果：** 第二段轨迹的接触峰值改变了隐藏状态，峰值消失后这种影响仍通过循环连接继续传递。因此最后输入虽然相同，两段轨迹的最后隐藏状态仍不完全相同。隐藏维度并不自动等同于“接触记忆”或“运动方向”；这里只能说固定权重把不同历史映射成了不同内部数值。

## 3. 顺序和 mask 都会改变记忆

循环更新不是对所有时间步求平均。把同一组观测倒序输入会形成不同记忆；在真实序列后补零时，如果没有 mask，补齐值也会继续修改隐藏状态。下面同时观察这两个问题。

In [ ]:
reversed_hidden = run_manual_rnn(release_sequence[::-1])  # 倒序读取完全相同的一组观测
padding = np.zeros((5, release_sequence.shape[1]))  # 构造五个只用于补齐长度的零时间步
padded_sequence = np.concatenate([release_sequence, padding], axis=0)  # 把补齐位置追加到真实轨迹之后
padded_mask = np.concatenate([np.ones(len(release_sequence), dtype=bool), np.zeros(len(padding), dtype=bool)])  # 标记真实位置和补齐位置
unmasked_hidden = run_manual_rnn(padded_sequence)  # 不使用 mask 让补齐值继续更新记忆
masked_hidden = run_manual_rnn(padded_sequence, padded_mask)  # 使用 mask 在真实轨迹结束后保持记忆
final_states = np.stack([release_hidden[-1], reversed_hidden[-1], unmasked_hidden[-1], masked_hidden[-1]])  # 收集四种处理方式的最后隐藏状态
state_labels = ["Correct order", "Reversed", "Padding unmasked", "Padding masked"]  # 使用通用英文字体准备四种最后状态的横轴标签
x_positions = np.arange(len(state_labels))  # 建立四组柱状图的横轴位置
bar_width = 0.34  # 设置两个隐藏维度柱子的宽度
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))  # 创建最终状态与补齐过程两个坐标轴
axes[0].bar(x_positions - bar_width / 2, final_states[:, 0], width=bar_width, label="Hidden 1")  # 绘制第一个隐藏维度的最终值
axes[0].bar(x_positions + bar_width / 2, final_states[:, 1], width=bar_width, label="Hidden 2")  # 绘制第二个隐藏维度的最终值
axes[0].set_xticks(x_positions, state_labels, rotation=18)  # 显示四种序列处理方式的标签
axes[0].set(title="Final memory by sequence handling", ylabel="Hidden-state value")  # 使用通用英文字体标注最终状态柱状图
axes[0].legend()  # 显示两个隐藏维度的图例
axes[1].plot(unmasked_hidden[:, 0], "o-", label="Without mask")  # 绘制补齐值继续改变记忆的过程
axes[1].plot(masked_hidden[:, 0], "s-", label="With mask")  # 绘制补齐阶段保持记忆的过程
axes[1].axvline(len(release_sequence) - 0.5, color="black", linestyle="--", label="Real sequence ends")  # 标出真实数据和补齐数据的边界
axes[1].set(title="Does padding update memory?", xlabel="Time step", ylabel="Hidden 1")  # 使用通用英文字体标注补齐对比图
axes[1].legend()  # 显示 mask 对比图的图例
fig.tight_layout()  # 调整两幅图的边距和文字位置
plt.show()  # 显示顺序与 mask 的影响

**怎样理解结果：** 倒序序列包含完全相同的观测值，却得到不同的最终隐藏状态，说明循环模型对先后顺序敏感。补齐未遮挡时，连续零输入仍会经过权重和偏置更新状态；使用 mask 后，真实序列结束处的记忆被保持下来。实际训练中，mask、episode 边界和时间对齐错误都会改变模型学到的历史含义。

**可以继续尝试：** 调小 `hidden_weights` 的对角线数值，观察接触峰值留下的影响是否更快消失；再调大这些数值，观察记忆保持与数值饱和之间的关系。